# Forecasting Experiment
Dieses Notebook enthält den Code zu den Forcasting Experimenten.


## Imports

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install ema-pytorch==0.2.1;pandas==1.5.0;scikit-learn==1.1.2;scipy==1.8.1;seaborn==0.12.2;tqdm==4.64.1;dm-control==1.0.12;dm-env==1.6;dm-tree==0.1.8;mujoco==2.3.4;gluonts==0.12.6;pyyaml==6.0
import os
import torch
import numpy as np
import sys
sys.path.insert(0,'/content/drive/MyDrive/Masterthesis')

from engine.solver import Trainer
from Utils.metric_utils import visualization
from Data.build_dataloader import build_dataloader
from Utils.io_utils import load_yaml_config, instantiate_from_config
from Models.interpretable_diffusion.model_utils import unnormalize_to_zero_to_one

from Utils.metric_utils import display_scores
from Utils.discriminative_metric import discriminative_score_metrics
from Utils.predictive_metric import predictive_score_metrics
from sklearn.preprocessing import MinMaxScaler
import itertools
import pandas as pd
import yaml
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt
from Utils.cross_correlation import CrossCorrelLoss
!pip install gluonts

import warnings

warnings.filterwarnings("ignore")

from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import Dataset, DataLoader
from gluonts.dataset.repository.datasets import get_dataset
from gluonts.dataset.multivariate_grouper import MultivariateGrouper
from Models.interpretable_diffusion.model_utils import normalize_to_neg_one_to_one

## Data Pre Processing

In [ ]:
#dataset paths
path_bluechip = "/content/drive/MyDrive/Masterthesis/Data/Datasets/bluechip_sp500.csv"
path_crossAsset = "/content/drive/MyDrive/Masterthesis/Data/Datasets/crossAsset_portfolio.csv"
#sequence lenghts
forcast_sequence_bluechip = 34
forcast_sequence_crossAsset = 34
#train lenghts
bluechip_train_length = 4564
crossAsset_train_length = 5298
#config paths
config_path_bluechip = '/content/drive/MyDrive/Masterthesis/Config/bluechip_sp500.yaml'
config_path_crossAsset = '/content/drive/MyDrive/Masterthesis/Config/crossAsset.yaml'

#current setting
path = path_bluechip
forcast_sequence = forcast_sequence_bluechip
train_length = bluechip_train_length
config =config_path_bluechip

In [ ]:

data_df = pd.read_csv(path)
#this functions slices the data into overlapping sequences
def Sequence_slicer(data, window):
      num_variables = data.shape[1]
      seq_length = window
      overlap_size = window-1
      num_points = data.shape[0]
      step_size = seq_length - overlap_size
      num_sequences = (num_points - seq_length) // step_size + 1
      trimmed_data = np.array([data[i:i + seq_length] for i in range(0, num_points - seq_length + 1, step_size)])

      return trimmed_data.reshape(num_sequences, seq_length, num_variables)

data = Sequence_slicer(data_df.values, forcast_sequence)

#This class generates the masking for the test data, depending on pred_length.
class Data(Dataset):
    def __init__(self, data,  pred_length, regular=True):
        super(Data, self).__init__()
        self.sample_num = data.shape[0]
        self.samples = data
        self.regular = regular
        self.mask = np.ones_like(data)
        self.mask[:, -pred_length:, :] = 0.
        self.mask = self.mask.astype(bool)

    def __getitem__(self, ind):
        x = self.samples[ind, :, :]
        if self.regular:
            return torch.from_numpy(x).float()
        mask = self.mask[ind, :, :]
        return torch.from_numpy(x).float(), torch.from_numpy(mask)

    def __len__(self):
        return self.sample_num

In [ ]:
#The class loads the settings from the configs
class Args_Example:
    def __init__(self) -> None:
        self.config_path = config
        self.save_dir = '/content/drive/MyDrive/Masterthesis/Data/Sequences/blueChip/samples' #Change the path if needed
        self.gpu = 0
        os.makedirs(self.save_dir, exist_ok=True)

args =  Args_Example()
configs = load_yaml_config(args.config_path)
#Use GPU if possible
device = torch.device(f'cuda:{args.gpu}' if torch.cuda.is_available() else 'cpu')


In [ ]:
#Find out the dimension of our data
feature_dim = np.shape(data)[-1]
seq_length = configs['model']['params']['seq_length']
#milestones are set during the training process (here every 1000 step).
#Since blueChip was trained for 15.000 epochs we use:
milestone = 15 ### corssAsset -> 10
forcast_length = 10

In [ ]:
def build_trainer(end, epoch, milestone):
  #This function is used to load the new observed data from the test dataset
  #into the model.

  #New oberved end point
  configs['dataloader']['train_dataset']['params']['end'] = end
  #configs['model']['params']['sampling_timesteps'] = 200
  #configs['solver']['max_epochs'] = epoch
  #configs['solver']['save_cycle'] = epoch
  train = data[:end].reshape(-1, data.shape[-1])
  #Scaling the data
  scaler = MinMaxScaler()
  train_scaled = normalize_to_neg_one_to_one(scaler.fit_transform(train))
  #masking the new data with the Data class
  train_dataset = Data(train_scaled.reshape(len(data[:end]), seq_length, data.shape[-1]) , pred_length = forcast_length)
  #specific data loading function, since we work with gpu's.
  dataloader = DataLoader(train_dataset, batch_size=64, shuffle=False, num_workers=0, drop_last=True, pin_memory=True, sampler=None)
  model = instantiate_from_config(configs['model']).to(device)
  #build the new model. The Trainier class is the actual Diffusionmodel implemented
  #by the Diffusion-TS paper.
  trainer = Trainer(config=configs, args=args, model=model, dataloader={'dataloader':dataloader})

  trainer.load(milestone = milestone)
  return trainer, scaler


def prediction_scaled(end,data,seq_length,scaler):
  #Scaling of the prediction part
  train = data[:end-seq_length]
  prediction = data[end-seq_length:end,:]
  prediction_scaled = scaler.transform(prediction).reshape(1,seq_length,np.shape(data)[-1])

  prediction_scaled = normalize_to_neg_one_to_one(prediction_scaled)
  return prediction_scaled

## Loop

In [ ]:
#This is the main generating loop.
for i in range(forcast_length, (len(data_df.values) - train_length)+10, forcast_length):
  trainer, scaler = build_trainer(train_length+i, 1 ,milestone)
  #if (i/forcast_length) % 10 == 0 and i != 0:
  #  trainer, scaler = build_trainer(train_length+i, 1 ,milestone)

  #scaling of the prediction part
  pred = prediction_scaled(train_length+i, data_df.values,seq_length,scaler)
  #Number of Sequences per forcast
  num_samples = 10000
  #Since we use GPU's we can do fancy multiple generation on the sequences at once
  pred = np.repeat(pred, num_samples, axis=0)
  #Again the Data class masks all of our 10.000 sequences at once
  prediction_dataset = Data(pred, regular=False, pred_length = forcast_length)
  #With the T4 GPU we can only handle 5.000 sequences at once, otherwise we run
  #out of VRAM.
  prediction_dataloader = torch.utils.data.DataLoader(prediction_dataset, batch_size=5000, shuffle=False, num_workers=0, pin_memory=True, sampler=None)
  samples = []
  #.restore is the forcasting function implemented by Diffusion-TS
  #We use 50 sampling_steps which are enought since we dont generate the whole 34
  #points, but the last 10 with the prior 24 as knowledge.
  synth_data, *_ = trainer.restore(prediction_dataloader, shape=[seq_length, feature_dim], sampling_steps=50)
  samples.append(synth_data)
  sample = np.concatenate(samples, axis=0)
  #scaling it back
  sample = scaler.inverse_transform(unnormalize_to_zero_to_one(sample.reshape(-1, feature_dim))).reshape(sample.shape)
  #just safe the generated part
  sample_np = sample[:,-forcast_length:,:].reshape(-1, feature_dim)
  #Saving it.
  filename = os.path.join(args.save_dir, f'synthetic_data_window_{int(i/forcast_length)}.csv')
  synth_df = pd.DataFrame(sample_np)
  synth_df.to_csv(filename, index=False)
  print(f"Generated and saved synthetic data for window {i/forcast_length} to {filename}")
